# 07 - Map Perturbation Changes onto Tree

In [ ]:
%autosave 0

In [ ]:
import os
import sys
import dill
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import plotting.settings
from adjustText import adjust_text
from dynaconf import Dynaconf
from tqdm import tqdm

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data

In [ ]:
import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from dataclasses import dataclass
from scipy.stats import f_oneway

def hierarchy_pos(G, root=None, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
    if not nx.is_tree(G):
        raise TypeError("hierarchy_pos can only be used on a tree.")

    if root is None:
        root = next(iter(nx.topological_sort(G))) if isinstance(G, nx.DiGraph) else random.choice(list(G.nodes))

    def recurse(node, width, vert_loc, xcenter, pos, parent=None):
        pos[node] = (xcenter, vert_loc)
        children = list(G.neighbors(node))
        if parent is not None and not isinstance(G, nx.DiGraph):
            children = [c for c in children if c != parent]

        if children:
            dx = width / len(children)
            x = xcenter - width / 2 - dx / 2
            for child in children:
                x += dx
                recurse(child, width=dx, vert_loc=vert_loc - vert_gap, xcenter=x, pos=pos, parent=node)
        return pos

    return recurse(root, width, vert_loc, xcenter, pos={})

def get_norm(values, mode):
    v = np.array(values)
    
    if mode == "diverging":
        vmax = max(abs(v.min()), abs(v.max()))
        return TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    else:
        return Normalize(vmin=v.min(), vmax=v.max())

def draw_modality_panel(G, pos, ax, tf, title, values_dict, cmap_name, mode):
    values = np.array(list(values_dict.values()))
    cmap = plt.cm.get_cmap(cmap_name)
    norm = get_norm(values, mode)

    # Prepare colorbar
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Draw edges
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        arrows=False,
        edge_color="gray",
        width=1.5,
        alpha=0.6
    )

    # Draw nodes
    node_colors = [values_dict[n] for n in G.nodes()]
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=900,
        node_color=node_colors,
        cmap=cmap,
        vmin=norm.vmin,
        vmax=norm.vmax,
        edgecolors="black",
        linewidths=0.7
    )

    # Labels
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)

    # Title
    ax.set_title(f"{tf} — {title}", fontsize=12, fontweight="bold")
    ax.axis("off")

    # Colorbar
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.03)
    cbar.ax.set_ylabel(title, rotation=90, fontsize=9)

def plot_tf_tree(G, pos, tf, modalities):
    fig, axes = plt.subplots(1, len(modalities), figsize=(6 * len(modalities), 6))

    for ax, (title, values_dict, cmap_name, mode) in zip(axes, modalities):
        draw_modality_panel(G, pos, ax, tf, title, values_dict, cmap_name, mode)

    plt.tight_layout()
    plt.show()
    

@dataclass
class TreeDEMapper:
    """Align a milestone tree with DE results."""
    G: nx.DiGraph
    de: pd.DataFrame        # must contain ["milestone", "feature", "coef", "padj"]

    def milestones_for_tf(self, tf: str) -> pd.DataFrame:
        """Return DE rows for a single TF (feature)."""
        return self.de[self.de["feature"] == tf].copy()

    def get_values(self, tf: str, value_col: str = "coef") -> dict:
        """
        Get a dict: milestone → value_col (e.g., "coef") for the TF.
        Missing milestones get value 0.
        """
        df_tf = self.milestones_for_tf(tf)
        out = {m: 0.0 for m in self.G.nodes()}   # default = 0 for missing
        for _, row in df_tf.iterrows():
            out[row["milestone"]] = row[value_col]
        return out

    def get_subgraph(self, tf: str) -> nx.DiGraph:
        """Subgraph containing only milestones present in DE results."""
        df_tf = self.milestones_for_tf(tf)
        keep = set(df_tf["milestone"])
        keep = [n for n in self.G.nodes() if n in keep]
        print(keep)
        return self.G.subgraph(keep).copy()
    
    def get_edge_differences(self, tf: str, value_col: str = "coef") -> dict:
        """
        Compute edge differences for the TF:
        diff(u -> v) = value(v) - value(u)

        Missing values are treated as 0.
        """
        # get node-level values (milestone → coef)
        values = self.get_values(tf, value_col=value_col)

        # work with TF-specific subgraph
        G_sub = self.get_subgraph(tf)

        edge_diff = {}
        for u, v in G_sub.edges():
            edge_diff[(u, v)] = values[v] - values[u]

        return edge_diff
    
    def get_all_edge_differences(self, value_col: str = "coef") -> pd.DataFrame:
        """
        Compute edge differences for ALL TFs along ALL edges,
        BUT ONLY for edges where BOTH nodes have actual DE results.

        Returns tidy DataFrame with columns:
        ["feature", "u", "v", "coef_u", "coef_v", "edge_diff"]
        """
        rows = []

        # precompute TF → rows
        all_features = self.de["feature"].unique()

        # group DE per feature for efficient lookup
        grouped = {
            tf: df_tf.set_index("milestone")
            for tf, df_tf in self.de.groupby("feature")
        }

        for tf in all_features:
            df_tf = grouped[tf]  # DE rows for this TF
            for u, v in self.G.edges():

                # only process edges if BOTH nodes exist in df_tf
                if u not in df_tf.index or v not in df_tf.index:
                    continue   

                coef_u = df_tf.loc[u, value_col]
                coef_v = df_tf.loc[v, value_col]
                diff = coef_v - coef_u

                rows.append({
                    "feature": tf,
                    "u": u,
                    "v": v,
                    "coef_u": coef_u,
                    "coef_v": coef_v,
                    "edge_diff": diff,
                })

        return pd.DataFrame(rows)

    
def plot_tf_tree_from_mapper(mapper, tf, value_col="coef", cmap="coolwarm", mode="diverging"):
    G_sub = mapper.get_subgraph(tf)
    values = mapper.get_values(tf, value_col=value_col)
    
    # Filter values dict to subgraph nodes only
    values = {k: values[k] for k in G_sub.nodes()}
    
    pos = hierarchy_pos(G_sub, root="NGN3_ ce", width=2, vert_gap=0.5)

    fig, ax = plt.subplots(figsize=(6, 6))
    draw_modality_panel(
        G_sub,
        pos,
        ax=ax,
        tf=tf,
        title=value_col,
        values_dict=values,
        cmap_name=cmap,
        mode=mode
    )
    plt.tight_layout()
    plt.show()
    
edges = [
    ("NGN3+ cells", "EC/X/D/I/K cells"),
    ("EC/X/D/I/K cells", "EC cells"),
    ("EC/X/D/I/K cells", "X/D/I/K cells"),
    ("X/D/I/K cells", "X cells"),
    ("X/D/I/K cells", "D/I/K cells"),
    ("D/I/K cells", "D cells"),
    ("D/I/K cells", "I/K cells"),
]

G = nx.DiGraph()
G.add_edges_from(edges)
pos = hierarchy_pos(G, root="NGN3+ cells", width=2.0, vert_gap=0.5, vert_loc=0, xcenter=0.5)

### Get branch DE results

In [ ]:
# Load conditions
with open('/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_conditions.txt', "r") as f:
    tf_perturbations = [line.strip() for line in f.readlines()]

In [ ]:
for tf_perturb in tf_perturbations:
    print(f"Processing TF perturbation: {tf_perturb}")
    
    try:
        # Get DE results for relevant tf
        output_dir = Path(settings.ANALYSIS.tables_dir) / "branch_de_results"   
        de_results = pd.read_csv(output_dir / f"de_results_{tf_perturb}_vs_control_per_milestone.tsv", sep="\t")

        # Get DE results for relevant columns
        df = de_results[["milestone", "padj", "coef", "feature"]].copy()

        # Subset to only significant results
        df = df[df["padj"] < 0.05].copy()

        # Pivot wider
        df = df.pivot(index="milestone", columns="feature", values="coef")

        # Add rows for missing milestones with NA values
        for milestone in G.nodes():
            if milestone not in df.index:
                df.loc[milestone] = [np.nan] * df.shape[1]
                
        # Order df by milestone order in graph
        milestone_order = list(G.nodes())
        df = df.reindex(milestone_order)

        # Export as lineage model
        lineage_model = {
            "graph": G,
            "positions": pos,
            "mean_coef": df,
            "node_order": milestone_order,
        } 

        # Save lineage model as a pickle file using dill
        with open(Path(settings.ANALYSIS.models_dir) / f"parse_{tf_perturb}_perturb_lineage_model.pkl", "wb") as f:
            dill.dump(lineage_model, f)
            
    except Exception as e:
        print(f"Error processing {tf_perturb}: {e}")
        continue